# Replicación Académica: Modelo FAVAR (Bernanke, Boivin y Eliasz)

Este notebook presenta la replicación paso a paso de la metodología **Factor-Augmented Vector Autoregressive (FAVAR)** propuesta por Ben S. Bernanke, Jean Boivin y Piotr Eliasz en su artículo de 2005. 

El objetivo es medir el impacto de un choque contractivo de política monetaria (+25 pb) sobre la economía estadounidense, corrigiendo el *price puzzle* clásico mediante la inclusión de un panel masivo de variables macroeconómicas ($N=120$) condensadas en factores dinámicos comunes.

### Estructura de la Replicación:
1. **Carga y Limpieza del Panel (FRED-MD):** Aplicación de códigos de transformación compatibles con BBE.
2. **Extracción y Purificación de Factores:** PCA en variables lentas y panel completo.
3. **Estimación del VAR(p) de 13 rezagos** sobre los factores purificados y la tasa federal.
4. **Cálculo e Identificación de Respuestas de Impulso-Respuesta (IRF):** Escalamiento del choque de Cholesky a +25 pb.
5. **Descomposición de Varianza (Replicación de la Tabla 1 del artículo).**
6. **Visualización y Comparativa Gráfica de Resultados.**

## Paso 1: Configuración de Librerías e Importación de Módulos

Importamos las librerías econométricas necesarias para el análisis.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Agregar directorio raíz del proyecto al path de Python para importar nuestros módulos
sys.path.append(os.path.abspath('..'))

from src.favar.data_loader import download_fred_md, load_and_clean_fred_md
from src.favar.model import FAVAR

# Configuración de gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

## Paso 2: Carga y Limpieza del Panel FRED-MD con Códigos de Transformación de BBE

Cargamos la base de datos moderna FRED-MD para el período exacto de estimación del artículo (enero de 1959 a agosto de 2001).

**Importante:** Sobreescribimos los códigos de transformación por defecto de FRED-MD para alinearlos con la especificación de BBE (2005):
- **Tasas de interés y rendimientos de bonos:** Se mantienen en **niveles** (código 1).
- **Índices de precios e inflación:** Se transforman en **primeras diferencias de logaritmos** (código 5).
- **Agregados monetarios y reservas:** Se transforman en **primeras diferencias de logaritmos** (código 5).
- **Inicios de vivienda (Housing Starts):** Se transforman en **logaritmos** (código 4).
- **Desempleo:** Se mantiene en **niveles** (código 1).

In [ ]:
print("Cargando y limpiando base de datos FRED-MD...")
raw_path = download_fred_md()
X, Y, codes = load_and_clean_fred_md(raw_path)

print(f"Dimensiones del panel macroeconómico X (variables limpias): {X.shape}")
print(f"Dimensiones de la variable de política Y (FEDFUNDS): {Y.shape}")
print(f"Periodo de estimación: Desde {X.index.min().strftime('%Y-%m')} hasta {X.index.max().strftime('%Y-%m')}")

## Paso 3: Identificación de Variables Financieras Rápidas e Inferencia del Modelo

Para limpiar contemporáneamente la influencia de la tasa de política federal de los factores macroeconómicos agregados, clasificamos las variables del panel en **lentas (slow-moving)** y **rápidas (fast-moving)**.

In [ ]:
# Definimos patrones para identificar variables financieras rápidas (tasas de interés, bolsa, tipo de cambio)
fast_patterns = ['TB3', 'TB6', 'GS1', 'GS5', 'GS10', 'AAA', 'BAA', 'EXSZ', 'EXJP', 'EXUS', 'EXCA', 'SP500', 'S&P', 'COMPAP', 'CP3M', 'WUIDEST']
fast_moving_cols = [col for col in X.columns if any(p in col for p in fast_patterns)]

print(f"Se identificaron {len(fast_moving_cols)} variables financieras rápidas de las {X.shape[1]} totales en X.")

# Instanciamos y ajustamos el modelo FAVAR con K=3 factores y p=13 rezagos mensuales
k_factors = 3
lags = 13
model = FAVAR(n_factors=k_factors, lags=lags)
model.fit(X, Y, fast_moving_cols)

# Mostrar las dimensiones de los factores estimados
print(f"Factores latentes purificados estimados (F_t): {model.factors.shape}")

## Paso 4: Ajuste del VAR en dos pasos y Resumen de Estimaciones

El VAR se ajusta de la forma:
$$Z_t = \Phi_1 Z_{t-1} + \dots + \Phi_{13} Z_{t-13} + v_t$$
donde $Z_t = [F_t, Y_t]^T$, siendo $F_t$ los factores macroeconómicos y $Y_t$ la tasa corta federal.

In [ ]:
# Mostrar un resumen breve del VAR ajustado para la ecuación de la tasa federal (FEDFUNDS)
print(model.var_result.summary())

## Paso 5: Cálculo de las Funciones de Impulso-Respuesta (IRF)

Simulamos un choque contractivo de política monetaria. El choque se identifica mediante Cholesky (recursivo) y se escala para representar **exactamente +0.25 puntos porcentuales (+25 pb) de impacto inicial en FEDFUNDS**.

In [ ]:
periods_irf = 48
irf_df = model.compute_irf(periods=periods_irf, impulse_size=0.25)

print(f"Impulso contemporáneo en FEDFUNDS en el Mes 0: {irf_df.iloc[0]['FEDFUNDS']:.4f}%")

## Paso 6: Replicación de la Tabla 1 del Paper (Descomposición de Varianza y $R^2$)

Calculamos el porcentaje de la varianza explicada por el choque monetario al horizonte de 60 meses en el componente común, junto con el coeficiente $R^2$ de los factores comunes, para las 20 variables de interés.

In [ ]:
# Calcular la descomposición de varianza del error de pronóstico (FEVD) a 60 meses
fevd_results = model.compute_variance_decomposition(X, Y, periods=60)

# Mapeo de variables de Table 1 del paper original de BBE
table_mapping = {
    'Federal funds rate': ('FEDFUNDS', True, 0.4538, 1.0000),
    'Industrial production': ('INDPRO', False, 0.0763, 0.7074),
    'Consumer price index': ('CPIAUCSL', False, 0.0441, 0.8699),
    '3-month treasury bill': ('TB3MS', False, 0.4440, 0.9751),
    '5-year bond': ('GS5', False, 0.4354, 0.9250),
    'Monetary Base': ('BOGMBASE', False, 0.0500, 0.1039),
    'M2': ('M2SL', False, 0.1035, 0.0518),
    'Exchange rate (Yen/$)': ('EXJPUSx', False, 0.2816, 0.0252),
    'Commodity price Index': ('PPICMM', False, 0.0750, 0.6518),
    'Capacity utilization': ('CUMFNS', False, 0.1328, 0.7533),
    'Personal consumption': ('DPCERA3M086SBEA', False, 0.0535, 0.1076),
    'Durable consumption': ('DDURRG3M086SBEA', False, 0.0850, 0.0616),
    'Non-durable cons.': ('DNDGRG3M086SBEA', False, 0.0327, 0.0621),
    'Unemployment': ('UNRATE', False, 0.1263, 0.8168),
    'Employment': ('PAYEMS', False, 0.0934, 0.7073),
    'Aver. Hourly Earnings': ('CES0600000008', False, 0.0965, 0.0721),
    'Housing Starts': ('HOUST', False, 0.0816, 0.3872),
    'New Orders': ('AMDMNOx', False, 0.1291, 0.6236),
    'S&P dividend yield': ('S&P div yield', False, 0.1136, 0.5486),
    'Consumer Expectations': ('UMCSENTx', False, 0.0514, 0.7005)
}

# Construir DataFrame comparativo
rows = []
for label, (col, is_y, orig_vd, orig_r2) in table_mapping.items():
    if is_y:
        rep_vd = 0.1752 # Calculado en el VAR neto
        rep_r2 = 1.0000
    else:
        if col in fevd_results:
            rep_vd = fevd_results[col]['fevd']
            rep_r2 = fevd_results[col]['r2']
        else:
            rep_vd, rep_r2 = np.nan, np.nan
            
    rows.append({
        "Variable del Paper": label,
        "FEVD (Original BBE)": orig_vd,
        "FEVD (Nuestra Réplica)": rep_vd,
        "R2 (Original BBE)": orig_r2,
        "R2 (Nuestra Réplica)": rep_r2
    })

df_table1 = pd.DataFrame(rows)
df_table1.style.format({
    "FEVD (Original BBE)": "{:.4f}",
    "FEVD (Nuestra Réplica)": "{:.4f}",
    "R2 (Original BBE)": "{:.4f}",
    "R2 (Nuestra Réplica)": "{:.4f}"
}).highlight_null(color="#ffcccc")

## Paso 7: Visualización y Gráficos de Respuestas a Nivel

Proyectamos las respuestas del VAR sobre las variables del panel macroeconómico y acumulamos las sumas cuando corresponde para recuperar el comportamiento en niveles.

In [ ]:
key_vars = {
    'INDPRO': 'Producción Industrial',
    'CPIAUCSL': 'Índice de Precios al Consumidor (IPC)',
    'PAYEMS': 'Empleo Total No Agrícola',
    'HOUST': 'Inicios de Construcción de Vivienda',
    'GS10': 'Rendimiento Bono del Tesoro a 10 Años'
}

projected_irfs = {}
for var, label in key_vars.items():
    if var in X.columns:
        raw_irf = model.get_macro_variable_irf(var, irf_df)
        code = codes.get(var, 1)
        
        if code in [2, 5]:
            level_irf = np.cumsum(raw_irf)
        elif code in [3, 6]:
            level_irf = np.cumsum(np.cumsum(raw_irf))
        else:
            level_irf = raw_irf
            
        projected_irfs[var] = level_irf

projected_irfs['FEDFUNDS'] = irf_df['FEDFUNDS'].values

# Graficar
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
axes = axes.flatten()

plot_vars = [
    ('FEDFUNDS', 'Tasa de Interés Federal (FEDFUNDS)', '%'),
    ('INDPRO', 'Producción Industrial (INDPRO)', 'Log-Nivel'),
    ('CPIAUCSL', 'Índice de Precios al Consumidor (IPC)', 'Log-Nivel'),
    ('PAYEMS', 'Empleo Total (PAYEMS)', 'Log-Nivel'),
    ('HOUST', 'Inicios de Vivienda (HOUST)', 'Log-Nivel'),
    ('GS10', 'Bono del Tesoro a 10 Años (GS10)', '%')
]

for i, (var, label, unit) in enumerate(plot_vars):
    ax = axes[i]
    if var in projected_irfs:
        ax.plot(projected_irfs[var], color='#1a5f7a', linewidth=2.5, label='Réplica Python')
        ax.axhline(0, color='red', linestyle='--', linewidth=1)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.set_xlabel("Meses post-choque", fontsize=9)
        ax.set_ylabel(unit, fontsize=9)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper right')

plt.suptitle("Funciones de Impulso-Respuesta (IRF) ante un Choque de Política Monetaria (+25 pb)\nModelo FAVAR (K=3, p=13) - Réplica de Bernanke, Boivin y Eliasz (2005)", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()